<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp02_Candidate_Elimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 2: Candidate-Elimination Algorithm
**Course:** ITA06 - Machine Learning (Lab)

## Aim
For a given set of training examples stored in a .CSV file, implement and demonstrate the Candidate-Elimination algorithm to output a description of the set of all hypotheses consistent with the training examples (the *version space*).

## Background
Candidate-Elimination maintains two boundaries: **S** (the most specific consistent hypotheses) and **G** (the most general consistent hypotheses). Positive examples generalise S and prune G; negative examples specialise G and prune S. Every hypothesis lying between S and G is consistent with the data - together they describe the whole **version space**.

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training (Candidate-Elimination)
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
Same EnjoySport data as Experiment 1, stored in a CSV file.

In [ ]:
data = [
    ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same",   "Yes"],
    ["Sunny", "Warm", "High",   "Strong", "Warm", "Same",   "Yes"],
    ["Rainy", "Cold", "High",   "Strong", "Warm", "Change", "No"],
    ["Sunny", "Warm", "High",   "Strong", "Cool", "Change", "Yes"],
]
columns = ["Sky", "AirTemp", "Humidity", "Wind", "Water", "Forecast", "EnjoySport"]
pd.DataFrame(data, columns=columns).to_csv("enjoysport.csv", index=False)
print("Saved enjoysport.csv")

## Step 2: Data Loading
Read the CSV file.

In [ ]:
df = pd.read_csv("enjoysport.csv")
df

## Step 3: Data Preprocessing
Clean the data, split into attributes and target, and find the **domain** (possible values) of every attribute - the algorithm needs it to specialise the general boundary.

In [ ]:
df = df.apply(lambda col: col.str.strip())
print("Missing values:", df.isnull().sum().sum())

attributes = list(df.columns[:-1])
X = [tuple(r) for r in df[attributes].values]
y = df["EnjoySport"].values

domains = [sorted(df[a].unique()) for a in attributes]
for a, d in zip(attributes, domains):
    print(f"{a:<10}: {d}")

## Step 4: Exploratory Data Analysis (EDA)
Inspect how attribute values are distributed across the two classes and how big the hypothesis space is.

In [ ]:
print(df.describe(include="all").T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), attributes):
    sns.countplot(data=df, x=col, hue="EnjoySport", ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

# Syntactically distinct hypotheses: each attribute can take any value or '?', plus the empty hypothesis
size = int(np.prod([len(d) + 1 for d in domains]))
print("Number of conjunctive hypotheses (without the empty one):", size)

## Step 5: Model Training (Candidate-Elimination)
Full implementation with explicit **S** and **G** sets. The boundaries are printed after each example.

In [ ]:
def consistent(h, x):
    """Does hypothesis h classify instance x as positive?"""
    return all(hv == "?" or hv == xv for hv, xv in zip(h, x))

def more_general_or_equal(h1, h2):
    """h1 >= h2 (h1 covers everything h2 covers)."""
    return all(a == "?" or (a != "0" and (a == b or b == "0")) for a, b in zip(h1, h2))

def min_generalization(s, x):
    return tuple(xv if sv == "0" else (sv if sv == xv else "?") for sv, xv in zip(s, x))

def min_specializations(g, x, domains, S):
    result = []
    for i, gv in enumerate(g):
        if gv == "?":
            for value in domains[i]:
                if value != x[i]:
                    new_h = g[:i] + (value,) + g[i + 1:]
                    if any(more_general_or_equal(new_h, s) for s in S):
                        result.append(new_h)
    return result

def candidate_elimination(X, y, domains, verbose=True):
    n = len(X[0])
    S = {("0",) * n}
    G = {("?",) * n}
    if verbose:
        print("Initial   S =", S, "\n          G =", G, "\n")
    for t, (x, label) in enumerate(zip(X, y), 1):
        if label == "Yes":                                   # positive example
            G = {g for g in G if consistent(g, x)}
            newS = set()
            for s in S:
                if consistent(s, x):
                    newS.add(s)
                else:
                    s2 = min_generalization(s, x)
                    if any(more_general_or_equal(g, s2) for g in G):
                        newS.add(s2)
            S = {s for s in newS if not any(s != o and more_general_or_equal(s, o) for o in newS)}
        else:                                                # negative example
            S = {s for s in S if not consistent(s, x)}
            newG = set()
            for g in G:
                if not consistent(g, x):
                    newG.add(g)
                else:
                    for g2 in min_specializations(g, x, domains, S):
                        newG.add(g2)
            G = {g for g in newG if not any(g != o and more_general_or_equal(o, g) for o in newG)}
        if verbose:
            print(f"After example {t} ({label}):")
            print("   S =", sorted(S))
            print("   G =", sorted(G), "\n")
    return S, G

S_final, G_final = candidate_elimination(X, y, domains)
print("FINAL  S boundary:", sorted(S_final))
print("FINAL  G boundary:", sorted(G_final))

## Step 6: Model Evaluation
To verify the result we **brute-force** the entire hypothesis space, keep only hypotheses consistent with all training examples, and check that this equals the set of hypotheses between S and G.

In [ ]:
from itertools import product

def consistent_with_data(h):
    return all(consistent(h, x) == (lab == "Yes") for x, lab in zip(X, y))

all_hypotheses = product(*[d + ["?"] for d in [list(d) for d in domains]])
brute_force_vs = {h for h in all_hypotheses if consistent_with_data(h)}

between = {h for h in product(*[d + ["?"] for d in [list(d) for d in domains]])
           if any(more_general_or_equal(g, h) for g in G_final)
           and any(more_general_or_equal(h, s) for s in S_final)}

print("Version space size (brute force):", len(brute_force_vs))
print("Version space size (from S and G):", len(between))
print("Both methods agree:", brute_force_vs == between)

print("\nAll hypotheses in the version space:")
for h in sorted(brute_force_vs):
    print("  ", h)

## Step 7: Prediction on New Data
Classify new instances by **voting** among all hypotheses of the version space. If every hypothesis agrees the answer is certain, otherwise the instance is ambiguous.

In [ ]:
def classify(x):
    votes = [consistent(h, x) for h in brute_force_vs]
    pos = sum(votes)
    return pos, len(votes) - pos

new_instances = [
    ("Sunny", "Warm", "Normal", "Strong", "Cool", "Change"),
    ("Rainy", "Cold", "Normal", "Strong", "Warm", "Same"),
    ("Sunny", "Warm", "Normal", "Strong", "Warm", "Same"),
    ("Sunny", "Cold", "Normal", "Strong", "Warm", "Same"),
]
for x in new_instances:
    pos, neg = classify(x)
    verdict = "Yes" if neg == 0 else ("No" if pos == 0 else "Ambiguous")
    print(f"{x} -> {pos} positive votes, {neg} negative votes => {verdict}")

## Conclusion
The final boundaries are **S = {<Sunny, Warm, ?, Strong, ?, ?>}** and **G = {<Sunny, ?, ?, ?, ?, ?>, <?, Warm, ?, ?, ?, ?>}**, describing 6 consistent hypotheses. Brute-force enumeration confirmed the result. Unlike FIND-S, Candidate-Elimination uses negative examples and gives the complete version space.